# Avendi V8-Y3 · Full Campaign (Kaggle 2×T4)
- Pinned commit: `57bce6bd5a17e96fb9856505c50f2ac2f9e9b123`
- Defaults: `OWNER_AUTHORIZED = True`, `CAMPAIGN_MODE = 'FULL'`, `SESSION_MINUTES = 660`, `ONE_GO = True` (executes all 88 jobs, 50 studies, sealed confirmation, and verified exports in one uninterrupted Run All).


In [ ]:
SOURCE_SHA = '57bce6bd5a17e96fb9856505c50f2ac2f9e9b123'
OWNER_AUTHORIZED = True
CAMPAIGN_MODE = 'FULL'
SESSION_MINUTES = 660
ONE_GO = True
RESUME_INPUT = None
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
DISCOVER_INPUTS = ['ELENDRIL_V8_Y1_FULL.zip', 'ELENDRIL_V8_Y1_EVIDENCE.zip', 'AVENDI_V8_Y2_EVIDENCE.zip']
MIRROR_WORKING = True
EXPORT_GLOBS = ['*.zip', '*.sha256', 'FINAL_STATUS.json', 'PENDING_QUEUE.json', 'RESULTS.md', 'METRICS.json']
import os, shutil, subprocess, sys
from pathlib import Path
repo = Path('/kaggle/working/anra_avendi_y3_src') if Path('/kaggle/working').is_dir() else Path('.')
if repo != Path('.') and not (repo / '.git').exists():
    if repo.exists(): shutil.rmtree(repo)
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(repo)])
if repo != Path('.'):
    if subprocess.run(['git', '-C', str(repo), 'cat-file', '-e', SOURCE_SHA + '^{commit}']).returncode != 0:
        subprocess.check_call(['git', '-C', str(repo), 'fetch', 'origin', '--tags'])
    subprocess.check_call(['git', '-C', str(repo), 'checkout', '--detach', SOURCE_SHA])
    head = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    assert head == SOURCE_SHA, f'HEAD mismatch: {head} != {SOURCE_SHA}'
    os.chdir(str(repo))
if str(Path.cwd()) not in sys.path: sys.path.insert(0, str(Path.cwd()))
os.environ['PLANTHIR_BACKEND_MODULES'] = 'experiments.avendi_y3.module'
if not OWNER_AUTHORIZED:
    print('NOT LAUNCHED: set OWNER_AUTHORIZED = True before running GPU launch.')
    subprocess.check_call([sys.executable, '-m', 'experiments.avendi_y3.module', '--mode', CAMPAIGN_MODE, '--session-minutes', str(SESSION_MINUTES), '--inventory-only'])
else:
    import torch
    assert torch.cuda.device_count() >= 2, f'Expected >=2 GPUs, found {torch.cuda.device_count()}'
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'doctor'])
    cdir = Path('work/preflight_canary'); cdir.mkdir(parents=True, exist_ok=True); spec_p = cdir / 'spec.json'
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'init', '--backend', 'avendi_y3', '--out', str(spec_p)])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'validate', '--spec', str(spec_p), '--canary'])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'run', '--spec', str(spec_p), '--work-root', str(cdir / 'cpu'), '--canary', '--device', 'cpu'])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'verify-bundle', '--bundle-dir', str(cdir / 'cpu' / 'result_bundle')])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'replay-bundle', '--bundle-dir', str(cdir / 'cpu' / 'result_bundle'), '--output', str(cdir / 'REPLAY_RECEIPT.json')])


In [ ]:
if not globals().get('OWNER_AUTHORIZED', False):
    print('Launch skipped while OWNER_AUTHORIZED = False.')
else:
    import os, shutil, subprocess, sys
    from pathlib import Path
    roots = []
    if Path('/kaggle/input').is_dir():
        for n in DISCOVER_INPUTS: roots.extend(str(p) for p in sorted(Path('/kaggle/input').rglob(n)))
        for ck in sorted(Path('/kaggle/input').rglob('Y1-t1_factorial-S3-F00-s7')):
            if ck.is_dir() and str(ck.parent) not in roots: roots.append(str(ck.parent))
    if RESUME_INPUT is None and Path('/kaggle/input').is_dir():
        rc = [str(p) for p in sorted(Path('/kaggle/input').rglob('AVENDI_Y3_FULL.zip')) if p.is_file()]
        if rc: RESUME_INPUT = rc[-1]
    work = Path('/kaggle/working/avendi_y3_run') if Path('/kaggle/working').is_dir() else Path('runs/avendi_y3_run')
    cmd = [sys.executable, '-m', 'experiments.avendi_y3.module', '--mode', CAMPAIGN_MODE, '--session-minutes', str(SESSION_MINUTES), '--work-root', str(work)]
    if not globals().get('ONE_GO', True): cmd.append('--no-one-go')
    if RESUME_INPUT: cmd.extend(['--resume-input', str(RESUME_INPUT)])
    if roots: cmd.extend(['--input-roots', ','.join(roots)])
    print('Launching:', ' '.join(cmd))
    subprocess.check_call(cmd)
    if MIRROR_WORKING and Path('/kaggle/working').is_dir():
        top_dir = Path('/kaggle/working')
        mp = sorted(p for p in (work / 'archives').iterdir() if p.is_file()) if (work / 'archives').is_dir() else []
        mp.extend(work / fn for fn in ('FINAL_STATUS.json', 'PENDING_QUEUE.json', 'RESULTS.md', 'METRICS.json') if (work / fn).is_file())
        for p in mp:
            dst = top_dir / p.name
            if (dst.exists() or dst.is_symlink()) and dst.resolve() == p.resolve(): continue
            if dst.exists() or dst.is_symlink(): dst.unlink()
            try: os.link(p, dst)
            except OSError: shutil.copy2(p, dst)
            print('EXPORTED:', dst.name, f'({dst.stat().st_size / (1024 * 1024):.2f} MiB)')


In [ ]:
LINK_GLOBS = ['*.zip', '*.sha256', 'FINAL_STATUS.json', 'PENDING_QUEUE.json', 'RESULTS.md', 'METRICS.json']
from pathlib import Path
try:
    from IPython.display import FileLink, display
except Exception:
    FileLink = None
wdir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('.')
for pat in LINK_GLOBS:
    for f in sorted(wdir.glob(pat)):
        if FileLink is not None:
            display(FileLink(str(f)))
        else:
            print('ARTIFACT:', f)
